In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import os
from datetime import datetime
from tensorflow import keras 

In [ ]:
df=pd.read_csv(r"C:\Users\VICTUS\.keras\datasets\jena_climate_2009_2016_extracted\jena_climate_2009_2016.csv")
df


In [ ]:
df.index=pd.to_datetime(df['Date Time'],format='%d.%m.%Y %H:%M:%S' )


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['Date Time'][:3000],df['T (degC)'][:3000],label="temp over time",color='red')
plt.plot(df['Date Time'][:3000],df['sh (g/kg)'][:3000],label="temp over time",color='black')
plt.plot(df['Date Time'][:3000],df['VPmax (mbar)'][:3000],label="temp over time",color='blue')

plt.show()

In [ ]:
numeric_data=df.select_dtypes(include=["float64","int64"])
plt.figure(figsize=(12,6))
sns.heatmap(numeric_data.corr(),cmap='coolwarm',annot=True)
plt.title("relation between data")
plt.show()

In [ ]:
df_numeric = df.drop(columns=['Date Time'])
df_numeric=df_numeric[5::6]
temprature =df_numeric["T (degC)"]
print(df_numeric.shape)

print(df_numeric.head())


In [ ]:
training_data_len=int(np.ceil(len(df_numeric)*0.95))
training_data=df_numeric[:training_data_len]
scaler=StandardScaler()
scaled_data_tr=scaler.fit_transform(df_numeric[:training_data_len])
print(scaled_data_tr.shape)



In [ ]:
x_train,y_train=[],[]
window_size=48
for i in range(window_size,len(training_data)):
    x_train.append(scaled_data_tr[i-window_size:i,:])
    y_train.append(scaled_data_tr[i,:])

x_train=np.array(x_train)
y_train=np.array(y_train) 

x_train=np.reshape(x_train,(x_train.shape[0],x_train.shape[1],scaled_data_tr.shape[1]))
print(x_train.shape,y_train.shape)

In [ ]:
#building the model
model=keras.models.Sequential()

#first layer 
model.add(keras.layers.LSTM(64,return_sequences=True,input_shape=(x_train.shape[1],x_train.shape[2])))
          

#second layer 
model.add(keras.layers.LSTM(64,return_sequences=False))

#third layer (DENSE LAYER)
model.add(keras.layers.Dense(128,activation="relu"))

#fourth layer (dropout regularization layer)
model.add(keras.layers.Dropout(0.5))

#output layer 
model.add(keras.layers.Dense(14))

model.summary()
model.compile(optimizer="adam",
                loss="mse",
                metrics=[keras.metrics.RootMeanSquaredError()])



In [ ]:
training =model.fit(x_train,y_train,epochs=5,batch_size=64)


In [ ]:
test_data=scaler.transform(df_numeric[training_data_len-window_size:])

x_test,y_test=[],df_numeric[training_data_len:]

for i in range(window_size,len(test_data)):
    x_test.append(test_data[i-window_size:i,:])

x_test=np.array(x_test)
x_test=np.reshape(x_test,(x_test.shape[0],x_test.shape[1],test_data.shape[1]))
predictions=model.predict(x_test)

final_predictions=scaler.inverse_transform(predictions)
print(final_predictions.shape,y_test.shape)

In [ ]:
#plotting
train=df_numeric[:training_data_len]
test=df_numeric[training_data_len:]


prediction_index = df_numeric.index[training_data_len:]
prediction_index = prediction_index[:len(final_predictions)]
print(len(prediction_index))
print(test.shape)
test=test.copy()


In [ ]:
feature_names = df_numeric.columns

for i, feature in enumerate(feature_names):

    plt.figure(figsize=(15,5))

    plt.plot(prediction_index,
             final_predictions[:, i],
             label="Predicted")

    plt.title(feature)
    plt.xlabel("Date")
    plt.ylabel(feature)
    plt.legend()

    plt.show()

In [ ]:
actual = y_test.values
feature_names = df_numeric.columns

for i, feature in enumerate(feature_names):

    plt.figure(figsize=(15,5))

    plt.plot(prediction_index,
             actual[:, i],
             label="Actual")

    plt.plot(prediction_index,
             final_predictions[:, i],
             label="Predicted")

    plt.title(feature)
    plt.xlabel("Date")
    plt.ylabel(feature)
    plt.legend()

    plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error


mae = mean_absolute_error(test,final_predictions)
individual_maes = mean_absolute_error(test, final_predictions, multioutput='raw_values')
error_df = pd.DataFrame({
    'Column Name': test.columns,
    'Scaled MAE': individual_maes
}).sort_values(by='Scaled MAE', ascending=False)

print(error_df.to_string(index=False))




In [ ]:
plt.figure(figsize=(12,6))

plt.plot(test.index,test['p (mbar'],label="Actual Temperature",color='black',linewidth=3)
plt.plot(test.index,final_predictions,label="Predicted Temperature",color='yellow',linestyle="dashed",linewidth=2)
plt.title("LSTM Temperature Forecasting (Actual vs Predicted) ", fontsize=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Temprature (C)", fontsize=12)
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
df_numeric.shape

In [ ]:
model.save("LSTM multivariate Model")